In [18]:
import pysam
import numpy as np
import pandas as pd
import re
import plotly.express as px
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [9]:
def get_overlap(a: tuple, b: tuple) -> int:
    """ Calculates the overlap between two intervals.

    Args:
        a (tuple): Interval a = (start, end)
        b (tuple): Interval b = (start, end)

    Returns:
        int: Overlap between a and b
    """        
    
    return max(0, min(a[1], b[1]) - max(a[0], b[0]) + 1)

In [10]:
def get_end_position(start: int, cigar: str) -> int:
    
    cigar_tuples = [(int(length), op) for length, op in re.findall(r'(\d+)([MIDNSHP=X])', cigar)]
    
    # Calculate the end position
    end = start - 1  # Because positions are inclusive
    for length, op in cigar_tuples:
        if op in ["M", "D", "N", "=", "X"]:
            end += length

    return end

In [11]:
def get_clipped_span(read: pysam.AlignedSegment) -> tuple:
    """ Calculates the span of a read that is clipped.

    Args:
        read (pysam.AlignedSegment): Sequencing Read

    Returns:
        tuple: Reference start and end of the clipped part of the read
    """        
    
    ref_start = read.reference_start
    ref_end = read.reference_end
    cigar = read.cigartuples
    
    # Beginnging of read is clipped
    if cigar[0][0] == 4 or cigar[0][0] == 5:
        return (ref_start - cigar[0][1], ref_start - 1)
    
    # End of read is clipped
    if cigar[-1][0] == 4 or cigar[-1][0] == 5:
        return (ref_end + 1, ref_end + cigar[-1][1])
    
    return None

In [12]:
def get_overlap_mate_bins(read, bins):
    
    conns = np.zeros(len(bins))
    
    if read.is_mapped:
        for i, bin in enumerate(bins):
            conns[i] += int(bool(get_overlap((read.next_reference_start, read.next_reference_start + 150), bin)))
    
    return conns

In [13]:
def calculate_read_based_features(bam: pysam.AlignmentFile, chrom: str, bin: int, bin_start: int, bin_stop: int, start:int, end:int, suffix: str) -> pd.Series:
    """ Calculates read-based features for a region.

    Args:
        bam (pysam.AlignmentFile): BAM file
        chrom (str): Chromosome name
        bin (int): bin number
        bin_start (int): Bin Start position
        bin_stop (int): Bin end position
        start (int): SV start position
        end (int): SV end position
        suffix (str): Suffix for column names

    Returns:
        pd.Series: Series containing the read-based features
    """        
    
    # Manually set chromosomal average, this is usually calculated
    baseline_insertsize_median = 400
    baseline_insertsize_mad = 50
    baseline_mapq_mean = 60
    baseline_mapq_std = 10
    
    insert_sizes = []
    mapqs = []
    all_reads = 0
    all_reads_extended = 0
    clipped_reads = 0
    split_reads = 0
    split_reads_conn = np.zeros(4-bin)
    disco_ff_reads = 0
    disco_ff_conn = np.zeros(4-bin)
    disco_rr_reads = 0
    disco_rr_conn = np.zeros(4-bin)
    disco_rf_reads = 0
    disco_rf_conn = np.zeros(4-bin)

    # Set bins for overlap computations
    bins = [(start, start + 50), (end - 50, end), (end + 1, end + 51)][bin-1:]

    # Define labels for binned split read features
    labels_split_reads_conn = [f'ill_splitreads_{suffix}_II', f'ill_splitreads_{suffix}_III', f'ill_splitreads_{suffix}_IV'][bin-1:]
    
    # Define labels for binned discordant read pair features
    labels_disco_conn = np.array([[f'disco_ff_{suffix}_II', f'disco_ff_{suffix}_III', f'disco_ff_{suffix}_IV'], 
                                  [f'disco_rr_{suffix}_II', f'disco_rr_{suffix}_III', f'disco_rr_{suffix}_IV'], 
                                  [f'disco_rf_{suffix}_II',f'disco_rf_{suffix}_III', f'disco_rf_{suffix}_IV']])[:,bin-1:]
    labels_disco_conn = list(labels_disco_conn.flatten())
    
    for read in bam.fetch(chrom, bin_start - 5, bin_stop + 5):
        
        if not read.is_unmapped and not read.is_duplicate and not read.is_qcfail:
            
            # Only consider reads that overlap with the region for which we want to calculate the features
            if not read.reference_end <= bin_start and not read.reference_start >= bin_stop:
                
                # Insert size
                insert_sizes.append(abs(read.template_length))
                
                # Mapping quality
                mapqs.append(read.mapping_quality)
                
                # Split-reads
                if read.has_tag('SA'):
                    split_reads += 1
                    supplementary_alignment = read.get_tag('SA').split(',')
                    sa_cigar = supplementary_alignment[3]
                    sa_chrom = supplementary_alignment[0]
                    sa_start = int(supplementary_alignment[1])
                    sa_end = get_end_position(sa_start, sa_cigar)
                    
                    if sa_chrom == chrom:
                        for i, bin in enumerate(bins):
                            split_reads_conn[i] += int(bool(get_overlap((sa_start, sa_end), bin)))
                     
                # Read orientation for inversions
                if read.is_reverse and read.mate_is_reverse:
                    disco_rr_reads += 1
                    disco_rr_conn += get_overlap_mate_bins(read, bins)
                    
                elif not read.is_reverse and not read.mate_is_reverse:
                    disco_ff_reads += 1
                    disco_ff_conn += get_overlap_mate_bins(read, bins)
                
                # Read orientation for duplications
                elif read.is_read1 and read.is_reverse and not read.mate_is_reverse and read.template_length>0:
                    disco_rf_reads += 1
                    disco_rf_conn += get_overlap_mate_bins(read, bins)
                    
                elif read.is_read1 and not read.is_reverse and read.mate_is_reverse and read.template_length<0:
                    disco_rf_reads += 1
                    disco_rf_conn += get_overlap_mate_bins(read, bins)
                    
                elif read.is_read2 and not read.is_reverse and read.mate_is_reverse and read.template_length<0:
                    disco_rf_reads += 1
                    disco_rf_conn += get_overlap_mate_bins(read, bins)
                    
                elif read.is_read2 and read.is_reverse and not read.mate_is_reverse and read.template_length>0:
                    disco_rf_reads += 1
                    disco_rf_conn += get_overlap_mate_bins(read, bins)  
                    
                all_reads += 1

            # For clipped reads, we needed to extend the region by 5 bp
            all_reads_extended += 1
            clip_span = get_clipped_span(read)
            
            if clip_span != None:
                overlap = get_overlap(clip_span, [bin_start, bin_stop])
                
                if overlap > 0:
                    clipped_reads += 1
        
    if all_reads > 0:
        # add 0.1 to avoid -inf values
        insertsize_mean = np.round(np.log2((np.mean(insert_sizes) + 0.1) / (baseline_insertsize_median + 0.1)), 3)
        insertsize_std = np.round(np.log2((np.std(insert_sizes) + 0.1) / (baseline_insertsize_mad + 0.1)), 3)

        mapping_quality_mean = np.round(np.log2((np.mean(mapqs) + 0.1) / (baseline_mapq_mean + 0.1)), 3)
        mapping_quality_std = np.round(np.log2((np.std(mapqs) + 0.1) / (baseline_mapq_std + 0.1)), 3)

        splitreads_proportion = np.round(split_reads / all_reads, 3)
        if split_reads > 0:
            split_reads_conn_proportion = np.round(split_reads_conn / split_reads, 3)
        else:
            split_reads_conn_proportion = split_reads_conn
        
        clippedreads_proportion = np.round(clipped_reads / all_reads_extended, 3)
        
        disco_ff_proportion = np.round(disco_ff_reads / all_reads, 3)
        disco_ff_conn_proportion = np.round(disco_ff_conn / disco_ff_reads, 3) if disco_ff_reads > 0 else disco_ff_conn
        disco_rr_proportion = np.round(disco_rr_reads / all_reads, 3)
        disco_rr_conn_proportion = np.round(disco_rr_conn / disco_rr_reads, 3) if disco_rr_reads > 0 else disco_rr_conn
        disco_rf_proportion = np.round(disco_rf_reads / all_reads, 3)
        disco_rf_conn_proportion = np.round(disco_rf_conn / disco_rf_reads, 3) if disco_rf_reads > 0 else disco_rf_conn

    else:
        # special case: no reads in region
        insertsize_mean = np.round(np.log2(0.1 / (baseline_insertsize_median + 0.1)), 3)
        insertsize_std = np.round(np.log2(0.1 / (baseline_insertsize_mad + 0.1)), 3)

        mapping_quality_mean = np.round(np.log2(0.1 / (baseline_mapq_mean + 0.1)), 3)
        mapping_quality_std = np.round(np.log2(0.1 / (baseline_mapq_std + 0.1)), 3)

        splitreads_proportion = 0
        split_reads_conn_proportion = split_reads_conn
        
        clippedreads_proportion = 0
        
        disco_ff_proportion = 0
        disco_ff_conn_proportion = disco_ff_conn
        disco_rr_proportion = 0
        disco_rr_conn_proportion = disco_rr_conn
        disco_rf_proportion = 0
        disco_rf_conn_proportion = disco_rf_conn

    values = [insertsize_mean, insertsize_std, mapping_quality_mean, mapping_quality_std, 
              splitreads_proportion, clippedreads_proportion, disco_ff_proportion, disco_rr_proportion, 
              disco_rf_proportion] + list(split_reads_conn_proportion) + list(disco_ff_conn_proportion) + list(disco_rr_conn_proportion) + list(disco_rf_conn_proportion)
    index = ['ill_isize_mean_' + suffix, 'ill_isize_std_' + suffix, 'ill_mapq_mean_' + suffix, 'ill_mapq_std_' + suffix, 
             'ill_splitreads_' + suffix, 'ill_clipreads_' + suffix, 'ill_disco_ff_' + suffix, 'ill_disco_rr_' + suffix, 
             'ill_disco_rf_' + suffix] + labels_split_reads_conn + labels_disco_conn
    
    return pd.Series(values, index=index)

In [36]:
# Set parameters
bam_filename = '/confidential/tGenVar/tech/illumina/snakemake_results/bam_hg38/17_08/bwa_mem.pe.sorted.mdup.bam'
bam = pysam.AlignmentFile(bam_filename, 'rb')

df_calls = pd.read_csv('17-08_hg38_DEL_DUP_confirmation_status.csv')
df_calls_annot = df_calls.copy()
#df_calls_annot = df_calls_annot[(df_calls_annot['sv_type'] == 'DUP') & (df_calls_annot['confirmation_status'] == 1) & (df_calls_annot['caller'] == 'lumpy') & (df_calls_annot['sv_len'] > 500)].reset_index(drop=True).head(100)
#df_calls_annot = df_calls_annot.groupby(['sv_type','confirmation_status']).sample(1000)

In [15]:
df_calls_annot.loc[:, ['ill_isize_mean_I', 'ill_isize_std_I', 'ill_mapq_mean_I', 'ill_mapq_std_I', 
                       'ill_splitreads_I', 'ill_clipreads_I', 'ill_disco_ff_I', 'ill_disco_rr_I', 
                       'ill_disco_rf_I', 'ill_splitreads_I_II', 'ill_splitreads_I_III', 'ill_splitreads_I_IV', 
                       'disco_rr_I_II', 'disco_rr_I_III', 'disco_rr_I_IV',  
                       'disco_ff_I_II', 'disco_ff_I_III', 'disco_ff_I_IV', 
                       'disco_rf_I_II', 'disco_rf_I_III', 'disco_rf_I_IV']] = df_calls_annot.apply(lambda x: calculate_read_based_features(bam, x['chrom'], 1, x['start'] - 51, x['start'] - 1, x['start'], x['end'], 'I'), axis=1, result_type='expand')


df_calls_annot.loc[:, ['ill_isize_mean_II', 'ill_isize_std_II', 'ill_mapq_mean_II', 'ill_mapq_std_II', 
                       'ill_splitreads_II', 'ill_clipreads_II', 'ill_disco_ff_II', 'ill_disco_rr_II', 
                       'ill_disco_rf_II', 'ill_splitreads_II_III', 'ill_splitreads_II_IV',
                       'disco_rr_II_III', 'disco_rr_II_IV',
                       'disco_ff_II_III', 'disco_ff_II_IV',
                       'disco_rf_II_III', 'disco_rf_II_IV']] = df_calls_annot.apply(lambda x: calculate_read_based_features(bam, x['chrom'], 2, x['start'], x['start'] + 50, x['start'], x['end'], 'II'), axis=1, result_type='expand')
      
df_calls_annot.loc[:, ['ill_isize_2mean_III', 'ill_isize_std_III', 'ill_mapq_mean_III', 'ill_mapq_std_III', 
                       'ill_splitreads_III', 'ill_clipreads_III', 'ill_disco_ff_III', 'ill_disco_rr_III', 
                       'ill_disco_rf_III', 'ill_splitreads_III_IV', 'disco_rr_III_IV', 'disco_ff_III_IV', 'disco_rf_III_IV']] = df_calls_annot.apply(lambda x: calculate_read_based_features(bam, x['chrom'], 3, x['end'] - 50, x['end'], x['start'], x['end'], 'III'), axis=1, result_type='expand')

df_calls_annot.loc[:, ['ill_isize_mean_IV', 'ill_isize_std_IV', 'ill_mapq_mean_IV', 'ill_mapq_std_IV', 
                       'ill_splitreads_IV', 'ill_clipreads_IV', 'ill_disco_ff_IV', 'ill_disco_rr_IV', 
                       'ill_disco_rf_IV']] = df_calls_annot.apply(lambda x: calculate_read_based_features(bam, x['chrom'], 4, x['end'] + 1, x['end'] + 51, x['start'], x['end'], 'IV'), axis=1, result_type ='expand')

In [26]:
# investigate difference between split-read features for confirmed and not confirmed calls
import matplotlib.pyplot as plt


df_calls_annot_del = df_calls_annot.loc[df_calls_annot['sv_type']=='DEL']
df_calls_annot_dup = df_calls_annot.loc[df_calls_annot['sv_type'] == 'DUP']

# melt dataframes into for plotting
df_calls_annot_del_melt = pd.melt(df_calls_annot_del[['confirmation_status', 'ill_splitreads_I_II', 'ill_splitreads_I_III',
                                                      'ill_splitreads_I_IV', 'ill_splitreads_II_III', 'ill_splitreads_II_IV', 'ill_splitreads_III_IV']], id_vars=['confirmation_status'])
df_calls_annot_dup_melt = pd.melt(df_calls_annot_dup[['confirmation_status', 'ill_splitreads_I_II', 'ill_splitreads_I_III',
                                                      'ill_splitreads_I_IV', 'ill_splitreads_II_III', 'ill_splitreads_II_IV', 'ill_splitreads_III_IV']], id_vars=['confirmation_status'])

# use plotly express to plot boxplots
fig = px.box(df_calls_annot_del_melt, x='variable', y='value', color='confirmation_status')
fig.update_layout(title='Split-reads for deletions', xaxis_title='Bin', yaxis_title='Proportion of split-reads')
fig.show()

fig = px.box(df_calls_annot_dup_melt, x='variable', y='value', color='confirmation_status')
fig.update_layout(title='Split-reads for duplications', xaxis_title='Bin', yaxis_title='Proportion of split-reads')
fig.show()


In [34]:
# investigate difference between binned discordant-read features for confirmed and not confirmed calls
import matplotlib.pyplot as plt
import seaborn as sns


df_calls_annot_del = df_calls_annot.loc[df_calls_annot['sv_type'] == 'DEL']
df_calls_annot_dup = df_calls_annot.loc[df_calls_annot['sv_type'] == 'DUP']

# melt dataframes into for plotting
df_calls_annot_del_melt = pd.melt(df_calls_annot_del[['confirmation_status', 
                                                      'disco_rf_I_II', 'disco_rf_I_III', 'disco_rf_I_IV',
                                                      'disco_rf_II_III', 'disco_rf_II_IV',
                                                      'disco_rf_III_IV']], id_vars=['confirmation_status'])
df_calls_annot_dup_melt = pd.melt(df_calls_annot_dup[['confirmation_status', 
                                                      'disco_rf_I_II', 'disco_rf_I_III', 'disco_rf_I_IV',
                                                      'disco_rf_II_III', 'disco_rf_II_IV',
                                                      'disco_rf_III_IV']], id_vars=['confirmation_status'])

fig = px.box(df_calls_annot_del_melt, x='variable', y='value', color='confirmation_status')
fig.update_layout(title='Discordant-reads for deletions', xaxis_title='Bin', yaxis_title='Proportion of discordant-reads')
fig.show()

fig = px.box(df_calls_annot_dup_melt, x='variable', y='value', color='confirmation_status')
fig.update_layout(title='Discordant-reads for duplications', xaxis_title='Bin', yaxis_title='Proportion of discordant-reads')
fig.show()

In [35]:
# investigate difference between binned discordant-read features for confirmed and not confirmed calls
import matplotlib.pyplot as plt


df_calls_annot_del = df_calls_annot.loc[df_calls_annot['sv_type'] == 'DEL']
df_calls_annot_dup = df_calls_annot.loc[df_calls_annot['sv_type'] == 'DUP']

# melt dataframes into for plotting
df_calls_annot_del_melt = pd.melt(df_calls_annot_del[['confirmation_status', 'ill_disco_rf_I', 'ill_disco_rf_II', 'ill_disco_rf_III', 'ill_disco_rf_IV']], id_vars=['confirmation_status'])
df_calls_annot_dup_melt = pd.melt(df_calls_annot_dup[['confirmation_status', 'ill_disco_rf_I', 'ill_disco_rf_II', 'ill_disco_rf_III', 'ill_disco_rf_IV',]], id_vars=['confirmation_status'])

fig = px.box(df_calls_annot_del_melt, x='variable', y='value', color='confirmation_status')
fig.update_layout(title='Discordant-reads for deletions', xaxis_title='Bin', yaxis_title='Proportion of discordant-reads')
fig.show()

fig = px.box(df_calls_annot_dup_melt, x='variable', y='value', color='confirmation_status')
fig.update_layout(title='Discordant-reads for duplications', xaxis_title='Bin', yaxis_title='Proportion of discordant-reads')
fig.show()